# Preliminary SOH Regression Analysis — Cell 2_009

**Student contribution:** Samina — baseline and regression models

This notebook performs a preliminary regression experiment using the pilot feature table created from `2_009.h5`. It compares a training-mean baseline, Ridge Regression, Support Vector Regression (SVR), and Random Forest Regression.

> **Important limitation:** This is a preliminary random train/test split from one battery cell. It is suitable for the Intermediate Status Report, but it is not the final unseen-cell evaluation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_PATH = Path("pilot_features_2_009.csv")
if not DATA_PATH.exists():
    candidates = list(Path(".").glob("**/pilot_features_2_009.csv"))
    if candidates:
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError("Place pilot_features_2_009.csv in the notebook folder.")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from {DATA_PATH}")
df.head()

## 1. Define the target and input features

The target is `soh_proxy_percent`. Capacity-derived columns are excluded because the SOH proxy was calculated from capacity. Including them would create target leakage and make the results misleading.

In [ ]:
TARGET = "soh_proxy_percent"
EXCLUDED = {
    TARGET, "capacity_estimate", "charge_capacity_max", "discharge_capacity_max",
    "test_group", "cell_id", "experiment", "schedule_file", "start_datetime"
}
feature_columns = [
    c for c in df.columns
    if c not in EXCLUDED and pd.api.types.is_numeric_dtype(df[c])
]
X = df[feature_columns].copy()
y = df[TARGET].copy()

print("Target:", TARGET)
print("Features used:", feature_columns)
print("Rows:", len(X))
print("Missing target values:", int(y.isna().sum()))

valid = y.notna()
X, y = X.loc[valid], y.loc[valid]
X.describe().T

## 2. Create a reproducible pilot split

An 80/20 random split is used for this preliminary experiment. The final project will additionally use cell-held-out and profile-held-out splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training SOH mean:", round(y_train.mean(), 3))

In [ ]:
models = {
    "Ridge Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "SVR": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=10.0, epsilon=0.1))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=200, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

def metrics_row(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

results = []
predictions = {}
mean_prediction = np.repeat(y_train.mean(), len(y_test))
results.append(metrics_row("Training-mean baseline", y_test, mean_prediction))
predictions["Training-mean baseline"] = mean_prediction

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append(metrics_row(name, y_test, pred))
    predictions[name] = pred

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df.round(4)

In [ ]:
results_df.round(4).to_csv("regression_results_2_009.csv", index=False)
print("Saved regression_results_2_009.csv")

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = results_df.set_index("Model")[["MAE", "RMSE"]]
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Preliminary regression comparison — cell 2_009")
ax.set_ylabel("Error in SOH percentage points")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig("regression_model_comparison_2_009.png", dpi=200)
plt.show()

In [ ]:
# Actual versus predicted values for the best-RMSE model
best_model = results_df.iloc[0]["Model"]
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, predictions[best_model], alpha=0.8)
lims = [min(y_test.min(), predictions[best_model].min()), max(y_test.max(), predictions[best_model].max())]
ax.plot(lims, lims, "k--", label="Perfect prediction")
ax.set_xlabel("Actual SOH proxy (%)")
ax.set_ylabel("Predicted SOH proxy (%)")
ax.set_title(f"Actual vs. predicted: {best_model}")
ax.legend()
plt.tight_layout()
plt.savefig("regression_actual_vs_predicted_2_009.png", dpi=200)
plt.show()

## 3. Report interpretation

The metric values above are preliminary because the experiment uses only one cell and a small test-level feature table. Lower MAE and RMSE indicate smaller prediction errors. A larger R² indicates that the model explains more variation in the test targets. These results should not be interpreted as evidence that one model will generalize to unseen battery cells until the multi-cell experiments are completed.

## 4. Reproducibility details

* Data: `2_009.h5`, represented by `pilot_features_2_009.csv`
* Target: `soh_proxy_percent`
* Split: 80% training and 20% testing
* Random state: 42
* Ridge alpha: 1.0
* SVR kernel: RBF; C=10.0; epsilon=0.1
* Random Forest: 200 trees; random state=42
* Metrics: MAE, RMSE, and R²

The final project will repeat the analysis with multiple cells and cell-held-out/profile-held-out splits.